In [1]:
# controle van mijn bestanden 

from pathlib import Path
import os

print("Beschikbare datasets:")
print(os.listdir("/kaggle/input"))

print("\nZoeken naar HMS-train.csv:")
for file in Path("/kaggle/input").rglob("train.csv"):
    print(file)

print("\nZoeken naar het baseline-script:")
for file in Path("/kaggle/input").rglob("hms_sklearn_baseline.py"):
    print(file)

Beschikbare datasets:
['notebooks', 'datasets', 'competitions']

Zoeken naar HMS-train.csv:
/kaggle/input/competitions/hms-harmful-brain-activity-classification/train.csv

Zoeken naar het baseline-script:
/kaggle/input/datasets/hannaleysen/gith-l/src/hms_sklearn_baseline.py


In [2]:
from pathlib import Path

for folder in Path("/kaggle/input").iterdir():
    print(f"\nDataset: {folder.name}")

    for file in folder.rglob("*"):
        if file.is_file() and (
            "baseline" in file.name.lower()
            or "optimal" in file.name.lower()
            or file.suffix.lower() in [".ipynb", ".py"]
        ):
            print("  ", file)


Dataset: notebooks
   /kaggle/input/notebooks/hannaleysen/basline-optimalisatie/__notebook__.ipynb

Dataset: datasets
   /kaggle/input/datasets/hannaleysen/gith-l/Kaggle_notebook.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/Test_push_pandas.py
   /kaggle/input/datasets/hannaleysen/gith-l/template.py
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-02-KNN.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-03-KNN+PCA.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-04-LDA.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-05-linear-regression-newton.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-06-MLP.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/HMS-experiment-07-MLP+SDG.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/hms-experiment-08-XGBoost-full-mode.ipynb
   /kaggle/input/datasets/hannaleysen/gith-l/notebooks/stap-1-baseli

In [3]:
# geoptimaliseerde baseline inlezen 


import nbformat
import ast
from pathlib import Path

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/hannaleysen/basline-optimalisatie/__notebook__.ipynb")
nb = nbformat.read(NOTEBOOK_PATH, as_version=4)

# Haal ALLEEN de functie-definities op uit cel 5 (zodat er geen 'df' wordt verwacht)
tree = ast.parse(nb.cells[5].source)
selected_nodes = [node for node in tree.body if isinstance(node, ast.FunctionDef)]

clean_tree = ast.Module(body=selected_nodes, type_ignores=[])
ast.fix_missing_locations(clean_tree)
exec(compile(clean_tree, "<optimized_features>", "exec"), globals())

print("Geoptimaliseerde featurefuncties veilig geladen!")

print("build_features:", build_features)
print("window_features:", window_features)
print("spectrogram_features:", spectrogram_features)
print("band_columns:", band_columns)

Geoptimaliseerde featurefuncties veilig geladen!
build_features: <function build_features at 0x7c2b74d3f920>
window_features: <function window_features at 0x7c2b74d3f7e0>
spectrogram_features: <function spectrogram_features at 0x7c2b74d3f560>
band_columns: <function band_columns at 0x7c2b74d3f4c0>


In [4]:
import inspect

print(inspect.signature(build_features))
print(inspect.signature(window_features))
print(inspect.signature(spectrogram_features))

(meta, spec_dir, is_test=False)
(spec, cols)
(path, windows, is_test)


In [5]:
from pathlib import Path
import importlib.util

scripts = list(Path("/kaggle/input").rglob("hms_sklearn_baseline.py"))

if len(scripts) != 1:
    raise FileNotFoundError(
        f"Verwacht 1 baseline-bestand, gevonden: {scripts}"
    )

SCRIPT_PATH = scripts[0]

spec = importlib.util.spec_from_file_location("hms_baseline", SCRIPT_PATH)
baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline)

print("Baseline geladen:", SCRIPT_PATH)

Baseline geladen: /kaggle/input/datasets/hannaleysen/gith-l/src/hms_sklearn_baseline.py


In [6]:
# eerst nog dingetjes ophalen uit de baseline 
import inspect

print("MODEL_NAMES:")
print(baseline.MODEL_NAMES)              # welke modellen vergelijkt de baseline momenteel

print("\nFunctie fit:")
print(inspect.getsource(baseline.fit))   # Hoe worden bestaande modellen getraind 

print("\nFunctie load_labels:")
print(inspect.getsource(baseline.load_labels))  # hoe worden de 6 doelklassen ingelezen

print("\nFunctie to_matrix:")
print(inspect.getsource(baseline.to_matrix))    # om exact dezelfde featureselectie als de baseline te gebruiken

MODEL_NAMES:
['logreg', 'hgb']

Functie fit:
def fit(name, X, Y, seed=42):
    """Fit model `name` on features X and soft labels Y (rows are vote distributions)."""
    Xe, ye, we = expand(X, Y)
    m = make_model(name, seed)
    if name == "logreg":   # in a Pipeline the weight must be routed to the final step
        m.fit(Xe, ye, logisticregression__sample_weight=we)
    else:
        m.fit(Xe, ye, sample_weight=we)
    return m


Functie load_labels:
def load_labels(data_dir, max_specs=None, seed=42):
    """Return one row per EEG with its spectrogram id, label offsets, patient id and vote distribution."""
    train = pd.read_csv(data_dir / "train.csv")

    # train.csv has several labelled sub-segments per eeg_id: sum their votes and keep the offset range.
    agg = {"spectrogram_id": "first", "spectrogram_label_offset_seconds": ["min", "max"], "patient_id": "first"}
    agg.update({c: "sum" for c in CLASSES})
    df = train.groupby("eeg_id").agg(agg)
    df.columns = ["spectrogra

In [7]:
import os
import time
import warnings
import ast
import nbformat
import numpy as np
import pandas as pd
import inspect

from pathlib import Path
from joblib import Parallel, delayed

In [8]:
# DATA INLADEN – NIEUW: één rij per (EEG, sub-segment) i.p.v. één rij per EEG
N_JOBS = 2
REGIONS = ["LL", "RL", "LP", "RP"]
BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 20)}

DATA_DIR = Path("/kaggle/input/competitions/hms-harmful-brain-activity-classification")
assert (DATA_DIR / "train.csv").exists(), "train.csv niet gevonden!"

CLASSES = baseline.CLASSES

# 1. Oude tabel (één rij per EEG, gemiddeld label). Alleen nog nodig om later eerlijk te kunnen vergelijken met oude score.
df_eeg = baseline.load_labels(DATA_DIR, max_specs=None, seed=42)

# 2. NIEUW: elke rij van train.csv = één voorbeeld met zijn eigen label
train = pd.read_csv(DATA_DIR / "train.csv")
train["num_votes"] = train[CLASSES].sum(axis=1)
train[CLASSES] = train[CLASSES].div(train["num_votes"], axis=0)   # stemmen -> kansverdeling

# 3. Per EEG maximaal 3 sub-segmenten bewaren: eerste, middelste en laatste
train = train.sort_values(["eeg_id", "eeg_sub_id"]).reset_index(drop=True)
n_per_eeg = train.groupby("eeg_id")["eeg_id"].transform("size")
pos = train.groupby("eeg_id").cumcount()
keep = (n_per_eeg <= 3) | (pos == 0) | (pos == (n_per_eeg - 1) // 2) | (pos == n_per_eeg - 1)
df = train[keep].reset_index(drop=True)
df["row_id"] = np.arange(len(df))
print(f"{len(train)} rijen in train.csv -> {len(df)} rijen gebruikt "
      f"({df.eeg_id.nunique()} EEGs, {df.patient_id.nunique()} patiënten)")


# 4. NIEUW: featurefuncties per rij (hergebruikt jouw window_features en band_columns)
def spectrogram_features_rows(path, windows):
    """windows: lijst van (row_id, offset_in_seconden)."""
    spec = pd.read_parquet(path)
    cols = band_columns(list(spec.columns[1:]))
    spec = spec.iloc[:, 1:].to_numpy(dtype=np.float32)
    rows = []
    for row_id, offset in windows:
        start = int(offset // 2)                      # 1 spectrogramrij = 2 s
        feats_row = window_features(spec[start:start + 300], cols)   # 10 minuten
        feats_row["row_id"] = row_id
        rows.append(feats_row)
    return rows


def build_features_rows(meta, spec_dir):
    t0 = time.time()
    jobs = [delayed(spectrogram_features_rows)(
                f"{spec_dir}/{sid}.parquet",
                list(zip(g.row_id, g.spectrogram_label_offset_seconds)))
            for sid, g in meta.groupby("spectrogram_id")]
    results = Parallel(n_jobs=N_JOBS, batch_size=32)(jobs)
    out = pd.DataFrame([r for rows in results for r in rows])
    print(f"{len(jobs)} spectrograms -> {len(out)} feature rows in {time.time() - t0:.0f}s")
    return out


X_df = build_features_rows(df, DATA_DIR / "train_spectrograms")

# 5. Controle van het aantal features
feats = [c for c in X_df.columns if c != "row_id"]
print("Aantal features:", len(feats))
assert len(feats) == 192, f"Verwacht 192 features, gevonden: {len(feats)}"

# 6. Labels en features samenvoegen
df = df.merge(X_df, on="row_id", how="inner")

# 7. Matrices (zelfde namen als vroeger, dus de volgende cellen blijven werken)
X = baseline.to_matrix(df, feats)
Y = df[CLASSES].to_numpy(dtype=np.float64)
groups = df["patient_id"].to_numpy()          # groeperen op patiënt blijft essentieel
num_votes = df["num_votes"].to_numpy()
eeg_ids = df["eeg_id"].to_numpy()

print("Vorm van X:", X.shape, "| Vorm van Y:", Y.shape)


106800 rijen in train.csv -> 36646 rijen gebruikt (17089 EEGs, 1950 patiënten)
11138 spectrograms -> 36646 feature rows in 505s
Aantal features: 192
Vorm van X: (36646, 192) | Vorm van Y: (36646, 6)


In [9]:
# Bibliotheken importeren 
import xgboost as xgb
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, ParameterGrid


In [10]:
# Soft labels geschikt maken voor XGBoost 
def expand_soft_labels(X, Y):
    n, k = Y.shape
    X_repeated = np.repeat(X, k, axis=0)
    y_repeated = np.tile(np.arange(k), n)
    weights = Y.reshape(-1)
    keep = weights > 0

    return (X_repeated[keep], y_repeated[keep], weights[keep])


In [11]:
# Bouw XGBoost Model 

XGB_PARAMS = {
    "objective": "multi:softprob",
    "num_class": 6,
    "eval_metric": "mlogloss",

    "tree_method": "hist",

    "max_depth": 4,
    "eta": 0.05,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "min_child_weight": 5,
    "lambda": 5.0,
    "alpha": 0.1,

    "seed": 42,
    "nthread": 2}

NUM_BOOST_ROUND = 500


def train_xgboost(X_train, Y_train, X_valid, Y_valid):
    # 1. Soft labels uitbreiden voor de trainingsdata
    X_train_exp, y_train_exp, train_weights = expand_soft_labels(X_train, Y_train)
    dtrain = xgb.DMatrix(X_train_exp, label=y_train_exp, weight=train_weights)
    # 2. Soft labels uitbreiden voor de validatiedata
    X_valid_exp, y_valid_exp, valid_weights = expand_soft_labels(X_valid, Y_valid)
    dvalid = xgb.DMatrix(X_valid_exp, label=y_valid_exp, weight=valid_weights)
    # 3. XGBoost trainen met early stopping
    model = xgb.train(params=XGB_PARAMS, dtrain=dtrain, num_boost_round=1500, evals=[(dtrain, "train"), (dvalid, "valid")], early_stopping_rounds=100, verbose_eval=100)
    return model

def predict_xgboost(model, X_valid):
    dvalid = xgb.DMatrix(X_valid)
    # Als early stopping gebruikt werd:
    if model.attr("best_iteration") is not None:
        predictions = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))
    # Als early stopping niet gebruikt werd:
    else:
        predictions = model.predict(dvalid)
        
    predictions = predictions.reshape(-1, 6)
    predictions = np.clip(predictions, 1e-15, 1.0)
    predictions /= predictions.sum(axis=1, keepdims=True)

    return predictions

In [12]:
# functie om KL-score te berekenen 
def kl_score(y_true, y_pred, eps=1e-15):
    """
    Bereken de gemiddelde KL-divergentie.
    Een lagere score is beter.
    """

    p = np.clip(y_pred, eps, 1.0)
    p = p / p.sum(axis=1, keepdims=True)

    t = np.asarray(y_true)

    terms = np.zeros_like(t, dtype=np.float64)
    mask = t > 0

    terms[mask] = (
        t[mask] * np.log(t[mask] / p[mask])
    )

    return np.mean(np.sum(terms, axis=1))

In [13]:
# Cross-validatie met early stopping

N_FOLDS = 3
xgb_oof = np.zeros_like(Y, dtype=np.float64)
fold_scores = []
best_iterations = []

cv = GroupKFold(n_splits=N_FOLDS)

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X, Y, groups), start=1):
    print(f"Start XGBoost fold {fold}/{N_FOLDS}")

    # Train- en validatiedata apart houden
    X_train = X[train_idx]
    Y_train = Y[train_idx]
    X_valid = X[valid_idx]
    Y_valid = Y[valid_idx]

    # Train met early stopping
    model = train_xgboost(X_train, Y_train, X_valid,Y_valid)

    # Beste aantal iteraties opslaan
    best_iterations.append(model.best_iteration + 1)

    # Voorspellingen maken met de beste iteratie
    predictions = predict_xgboost(model, X_valid)
    xgb_oof[valid_idx] = predictions
    score = kl_score(Y_valid, predictions)
    fold_scores.append(score)

    print(f"Fold {fold} KL: {score:.6f}")
    print(f"Beste iteratie: {model.best_iteration + 1}")

print("Gemiddelde fold KL:", np.mean(fold_scores))
print("Totale OOF KL:", kl_score(Y, xgb_oof))
print("Gemiddelde beste iteratie:", np.mean(best_iterations))


# --- Evaluatie ---
hi = num_votes >= 10
print("\nKL per rij, alle rijen      :", kl_score(Y, xgb_oof))
print("KL per rij, >=10 stemmen    :", kl_score(Y[hi], xgb_oof[hi]))

# Vergelijkbaar met OUDE score: voorspellingen per EEG samenvoegen + vergelijken met oude gemiddelde EEG-label.
oof_df = pd.DataFrame(xgb_oof * num_votes[:, None], columns=CLASSES)
oof_df["eeg_id"] = eeg_ids
pred_eeg = oof_df.groupby("eeg_id")[CLASSES].sum()
pred_eeg = pred_eeg.div(pred_eeg.sum(axis=1), axis=0)
true_eeg = df_eeg.set_index("eeg_id").loc[pred_eeg.index, CLASSES].to_numpy()
print("KL per EEG (vergelijk met 0.897):", kl_score(true_eeg, pred_eeg.to_numpy()))

Start XGBoost fold 1/3
[0]	train-mlogloss:1.63093	valid-mlogloss:1.65120
[100]	train-mlogloss:1.08645	valid-mlogloss:1.28904
[200]	train-mlogloss:0.96385	valid-mlogloss:1.26055
[300]	train-mlogloss:0.88381	valid-mlogloss:1.25270
[400]	train-mlogloss:0.82185	valid-mlogloss:1.25264
[480]	train-mlogloss:0.78088	valid-mlogloss:1.25357
Fold 1 KL: 0.893644
Beste iteratie: 381
Start XGBoost fold 2/3
[0]	train-mlogloss:1.63562	valid-mlogloss:1.63784
[100]	train-mlogloss:1.08209	valid-mlogloss:1.32061
[200]	train-mlogloss:0.96098	valid-mlogloss:1.30121
[300]	train-mlogloss:0.88470	valid-mlogloss:1.30072
[331]	train-mlogloss:0.86533	valid-mlogloss:1.30065
Fold 2 KL: 0.966651
Beste iteratie: 232
Start XGBoost fold 3/3
[0]	train-mlogloss:1.63158	valid-mlogloss:1.64533
[100]	train-mlogloss:1.07913	valid-mlogloss:1.37876
[200]	train-mlogloss:0.95902	valid-mlogloss:1.35959
[300]	train-mlogloss:0.88021	valid-mlogloss:1.35385
[400]	train-mlogloss:0.81890	valid-mlogloss:1.35510
Fold 3 KL: 0.995073
Beste

In [14]:
# ===== FINAL XGBoost: fit op ALLE trainingsdata, voorspel test, schrijf submission.csv =====

# 1. Model trainen op alle trainingsdata (X en Y zijn al de volledige set)
final_rounds = int(np.median(best_iterations))
X_exp, y_exp, weights = expand_soft_labels(X, Y)
dtrain = xgb.DMatrix(X_exp, label=y_exp, weight=weights)
final_model = xgb.train(params=XGB_PARAMS, dtrain=dtrain, num_boost_round=final_rounds, verbose_eval=False)

# 2. Testfeatures (zelfde stappen als in de baseline)
test = pd.read_csv(DATA_DIR / "test.csv")
T_df = build_features(test, DATA_DIR / "test_spectrograms", is_test=True)
T_df = test[["eeg_id"]].merge(T_df, on="eeg_id", how="left")
T = baseline.to_matrix(T_df, feats)
assert T.shape[1] == 192


# 3. Voorspellen en submission schrijven
probs = predict_xgboost(final_model, T)

sub = pd.DataFrame({"eeg_id": T_df["eeg_id"]})
sub[baseline.CLASSES] = probs

assert len(sub) == len(test)
assert not sub.isna().any().any()
assert np.allclose(sub[baseline.CLASSES].sum(axis=1), 1.0)

sub.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission.csv", sub.shape)
sub.head()

1 spectrograms -> 1 feature rows in 0s
Saved submission.csv (1, 7)


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,3911565283,0.12653,0.038093,0.00366,0.338438,0.060792,0.432487
